**Automated script**

In [0]:
%pip install xgboost

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
# ========================================================
# Imports
# ========================================================
import pyspark.sql.functions as F
from pyspark.sql import Window
import joblib
import xgboost as xgb
import pandas as pd
from delta.tables import DeltaTable
from sklearn.base import BaseEstimator, TransformerMixin

# ========================================================
# 0. Custom Transformer (Required by Saved Model)
# ========================================================
class ToNumeric(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for c in self.columns:
            if c in X.columns:
                X[c] = pd.to_numeric(X[c], errors="coerce")
        return X

# ========================================================
# 1. Load Regression Model + Preprocess
# ========================================================
artifact_dir = "/Workspace/Users/suhani.thakur@hp.com/Shared/xgb_checkpoints"

xgb_loaded = xgb.XGBRegressor()
xgb_loaded.load_model(f"{artifact_dir}/best_model1.json")
preprocess_loaded = joblib.load(f"{artifact_dir}/preprocess_tuned.pkl")
_cat = list(preprocess_loaded.transformers[0][2])
_num = list(preprocess_loaded.transformers[1][2])

# ========================================================
# 2. Load SNI Label Model
# ========================================================
sni_label_model_path = "/Workspace/Users/suhani.thakur@hp.com/Shared/artifacts/model.joblib"
sni_label_model = joblib.load(sni_label_model_path)

_label_preproc = sni_label_model.named_steps["preprocess"]
_label_feature_cols = []
for _, _, cols in _label_preproc.transformers:
    _label_feature_cols.extend(cols)

# ========================================================
# 3. Build Required Column List BEFORE Loading Data
#    (OPT-2: Column Pruning — know what you need upfront)
# ========================================================
source_table = (
    "supplychain.shipped_not_invoiced."
    "fact_enterprise_shipped_not_invoiced_legacy_combined_reporting_view"
)
target_table = (
    "innovation_supplychain_stg.sni_otc."
    "sni_prediction_regression_classification_v2"
)

id_cols = [
    "Sales_Order_Identifier",
    "Sales_Order_Line_Item_Identifier",
    "Shipment_Identifier",
    "Shipment_Line_Item_Identifier",
    "Shipment_Date"
]

feature_cols = _cat + _num

_date_cols_for_engineering = [
    "Sales_Order_Header_Create_Date",
    "Proof_Of_Delivery_Date",
    "Planned_Shipment_Date",
    "planned_invoice_date",
    "Crdd",
    "Tdd"
]

# Combine all needed columns (+ Dm_Date for dedup, + Dm_Aging_Days as target)
all_needed_cols = list(dict.fromkeys(
    id_cols
    + feature_cols
    + _label_feature_cols
    + _date_cols_for_engineering
    + ["Dm_Aging_Days", "Dm_Date"]
))

# ========================================================
# 4. Load Source with Column Pruning + Early Date Filter
#    OPT-1: Shipment_Date filter BEFORE window function
#    OPT-2: Select only needed columns at read time
# ========================================================
today = F.current_date()
yesterday = F.date_sub(today, 1)

# Read schema once to validate column names exist
_src = spark.table(source_table)
_available = set(_src.columns)
select_cols = [c for c in all_needed_cols if c in _available]

sni_df = (
    _src
    .select(select_cols)                                    # OPT-2: column pruning
    .filter(F.col("Shipment_Date").isin(yesterday, today))  # OPT-1: filter first
)

# OPT-4: Capture source row count on already-filtered data
# (avoids expensive re-scan of full source table in audit cell)
_source_row_count = sni_df.count()
print(f"\u25b6 Source rows (pre-dedup): {_source_row_count:,}")

# ========================================================
# 5. Latest Snapshot Dedup (now runs on tiny subset)
#    OPT-3: Window operates on filtered data only instead
#    of the entire table. If Dm_Date always equals
#    current_date() for latest snapshots, this can be
#    further simplified to:
#       .filter(F.col("Dm_Date") == F.current_date())
# ========================================================
w = Window.partitionBy(
    "Sales_Order_Identifier",
    "Sales_Order_Line_Item_Identifier"
)

sni_df = (
    sni_df
    .withColumn("max_snapshot_date", F.max("Dm_Date").over(w))
    .filter(F.col("Dm_Date") == F.col("max_snapshot_date"))
    .drop("max_snapshot_date")
)

# ========================================================
# 5b. Capture eligible rows (post-dedup, pre-change-detection)
# ========================================================
_eligible_row_count = sni_df.count()
print(f"\u25b6 Eligible rows (post-dedup): {_eligible_row_count:,}")

# ========================================================
# 6. CHANGE-DETECTION — Feature Fingerprint
# ========================================================
_hash_input_cols = sorted(set(
    feature_cols
    + _label_feature_cols
    + _date_cols_for_engineering
))
_hash_input_cols = [c for c in _hash_input_cols if c in sni_df.columns]

sni_df = sni_df.withColumn(
    "feature_hash",
    F.md5(F.concat_ws(
        "||",
        *[F.coalesce(F.col(c).cast("string"), F.lit("__NULL__"))
          for c in _hash_input_cols]
    )),
)

# --- compare with target table ---
if spark.catalog.tableExists(target_table):
    _existing = (
        spark.table(target_table)
        .select(*id_cols, F.col("feature_hash").alias("_prev_hash"))
    )
    _src_cols = sni_df.columns
    sni_df = (
        sni_df.alias("src")
        .join(_existing.alias("tgt"), id_cols, "left")
        .filter(
            F.col("_prev_hash").isNull()
            | (F.col("_prev_hash") != F.col("src.feature_hash"))
        )
        .select([F.col(f"src.{c}") for c in _src_cols])
    )

# ========================================================
# 6b. Early exit when nothing changed
# ========================================================
df_inference = sni_df.cache()
_rows_to_predict = df_inference.count()
print(f"\u25b6 Rows requiring prediction: {_rows_to_predict}")

if _rows_to_predict == 0:
    print("\u2705 No new or changed rows \u2014 skipping prediction & merge.")
    dbutils.notebook.exit("NO_CHANGES")

# ========================================================
# 7. Date Casting
# ========================================================
for d in _date_cols_for_engineering:
    if d in df_inference.columns:
        df_inference = df_inference.withColumn(d, F.to_timestamp(F.col(d)))

# ========================================================
# 7b. Engineer time-delta features for the label model
# ========================================================
if {"Shipment_Date", "Sales_Order_Header_Create_Date"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_order_to_shipment",
        F.datediff(F.col("Shipment_Date"), F.col("Sales_Order_Header_Create_Date")).cast("double")
    )
if {"Proof_Of_Delivery_Date", "Shipment_Date"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_shipment_to_pod",
        F.datediff(F.col("Proof_Of_Delivery_Date"), F.col("Shipment_Date")).cast("double")
    )
if {"Shipment_Date", "Planned_Shipment_Date"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_planned_vs_actual_shipment",
        F.datediff(F.col("Shipment_Date"), F.col("Planned_Shipment_Date")).cast("double")
    )
if {"planned_invoice_date", "Sales_Order_Header_Create_Date"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_order_to_planned_invoice",
        F.datediff(F.col("planned_invoice_date"), F.col("Sales_Order_Header_Create_Date")).cast("double")
    )
if {"Crdd", "Tdd"} <= set(df_inference.columns):
    df_inference = df_inference.withColumn(
        "days_crdd_minus_tdd",
        F.datediff(F.col("Crdd"), F.col("Tdd")).cast("double")
    )
if {"Actual_Quantity_Delivered", "Shipment_Quantity"} <= set(df_inference.columns):
    df_inference = (
        df_inference
        .withColumn("delivered_ratio",
            F.when(F.col("Shipment_Quantity").isNull() | (F.col("Shipment_Quantity") == 0), F.lit(0.0))
             .otherwise(F.col("Actual_Quantity_Delivered") / F.col("Shipment_Quantity"))
        )
        .withColumn("delivered_ratio",
            F.when(F.col("delivered_ratio") < 0, 0.0)
             .when(F.col("delivered_ratio") > 5, 5.0)
             .otherwise(F.col("delivered_ratio"))
        )
    )

# ========================================================
# 8. Pandas Conversion + Predictions
# ========================================================
pdf = df_inference.toPandas()

# Convert Timestamp columns in feature sets to days since 2021-01-01
# (matches datediff approach used during model training)
for c in feature_cols + _label_feature_cols:
    if c in pdf.columns and pd.api.types.is_datetime64_any_dtype(pdf[c]):
        pdf[c] = (pdf[c] - pd.Timestamp("2021-01-01")).dt.days.astype(float)

# ---------- Regression ----------
X_reg = pdf[feature_cols]
X_enc = preprocess_loaded.transform(X_reg)
pdf["Predicted_Dm_Aging_Days"] = xgb_loaded.predict(X_enc)

# ---------- Classification ----------
X_label = pdf[_label_feature_cols]
pdf["SNI_Label"] = sni_label_model.predict(X_label)

# ========================================================
# 9. Convert Back to Spark
# ========================================================
df_predicted = spark.createDataFrame(
    pdf[
        id_cols +
        ["Dm_Aging_Days", "Predicted_Dm_Aging_Days", "SNI_Label", "feature_hash"]
    ]
).dropDuplicates(id_cols)

# Add prediction_date and prediction_timestamp
df_predicted = (
    df_predicted
    .withColumn("prediction_date", F.current_date())
    .withColumn("prediction_timestamp", F.current_timestamp())
)

# --------------------------------------------------------
# 9b. Ensure prediction_timestamp column exists in target (one-time)
# --------------------------------------------------------
if spark.catalog.tableExists(target_table):
    _tgt_cols = {f.name for f in spark.table(target_table).schema.fields}
    if "prediction_timestamp" not in _tgt_cols:
        spark.sql(
            f"ALTER TABLE {target_table} "
            f"ADD COLUMN (prediction_timestamp TIMESTAMP "
            f"COMMENT 'Exact time of prediction run')"
        )

# --------------------------------------------------------
# 10. MERGE (UPSERT) into Delta Table
#     prediction_timestamp is set here — no separate UPDATE needed
# --------------------------------------------------------
_merge_update_cols = ["Dm_Aging_Days", "Predicted_Dm_Aging_Days", "SNI_Label", "feature_hash", "prediction_date", "prediction_timestamp"]
_merge_insert_cols = id_cols + _merge_update_cols

if not spark.catalog.tableExists(target_table):
    (
        df_predicted
        .write
        .format("delta")
        .saveAsTable(target_table)
    )
else:
    delta = DeltaTable.forName(spark, target_table)

    (
        delta.alias("t")
        .merge(
            df_predicted.alias("s"),
            """
            t.Sales_Order_Identifier = s.Sales_Order_Identifier AND
            t.Sales_Order_Line_Item_Identifier = s.Sales_Order_Line_Item_Identifier AND
            t.Shipment_Identifier = s.Shipment_Identifier AND
            t.Shipment_Line_Item_Identifier = s.Shipment_Line_Item_Identifier AND
            t.Shipment_Date = s.Shipment_Date
            """
        )
        .whenMatchedUpdate(set={c: f"s.{c}" for c in _merge_update_cols})
        .whenNotMatchedInsert(values={c: f"s.{c}" for c in _merge_insert_cols})
        .execute()
    )

/local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.10/site-packages/xgboost/sklearn.py:782: UserWarning: Loading a native XGBoost model with Scikit-Learn interface.
  warnings.warn("Loading a native XGBoost model with Scikit-Learn interface.")


▶ Rows requiring prediction: 4070
✅ Merged 4070 predicted rows into innovation_supplychain_stg.sni_otc.sni_prediction_regression_classification_v2


In [0]:
# ========================================================
# POST-PREDICTION: Audit Logging
# Runs AFTER the prediction merge — captures run metrics
# and appends an audit record.
# ========================================================
import uuid
import pyspark.sql.functions as F

_audit_table = "innovation_supplychain_stg.sni_otc.sni_prediction_run_log"

# --------------------------------------------------------
# 1. Gather run metadata + detect run type
#    Check for workspace (interactive) first;
#    default to scheduled_job in else
# --------------------------------------------------------
try:
    _ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    _tags = _ctx.tags()
    _notebook_path = _tags.get("notebookPath").getOrElse(None)
    _cluster_id    = _tags.get("clusterId").getOrElse(None)
    _job_id        = str(_tags.get("jobId").getOrElse(""))
    if _job_id == "":
        _run_type = "interactive"
    else:
        _run_type = "scheduled_job"
except Exception:
    _notebook_path = None
    _cluster_id    = None
    _run_type      = "scheduled_job"

_run_id = str(uuid.uuid4())

print(f"\u25b6 Detected run type: {_run_type}")

# --------------------------------------------------------
# 2. Ensure audit log table exists (clean schema)
# --------------------------------------------------------
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {_audit_table} (
        run_id              STRING,
        run_timestamp       TIMESTAMP,
        run_type            STRING,
        source_row_count    LONG,
        eligible_row_count  LONG,
        changed_row_count   LONG,
        predicted_row_count LONG,
        prediction_date     DATE,
        notebook_path       STRING,
        cluster_id          STRING
    ) USING DELTA
    COMMENT 'Audit log for SNI batch prediction runs'
""")

# Add run_type column if table already existed without it
_log_cols = {f.name for f in spark.table(_audit_table).schema.fields}
if "run_type" not in _log_cols:
    spark.sql(
        f"ALTER TABLE {_audit_table} "
        f"ADD COLUMN (run_type STRING COMMENT 'interactive or scheduled_job')"
    )

# --------------------------------------------------------
# 3. Insert audit record
#    _source_row_count & _eligible_row_count from Cell 3
# --------------------------------------------------------
_nb_path_sql = f"'{_notebook_path}'" if _notebook_path else "NULL"
_cl_id_sql   = f"'{_cluster_id}'"    if _cluster_id   else "NULL"

spark.sql(f"""
    INSERT INTO {_audit_table}
    (run_id, run_timestamp, run_type, source_row_count, eligible_row_count,
     changed_row_count, predicted_row_count, prediction_date,
     notebook_path, cluster_id)
    VALUES (
        '{_run_id}',
        current_timestamp(),
        '{_run_type}',
        {_source_row_count},
        {_eligible_row_count},
        {_rows_to_predict},
        {_rows_to_predict},
        current_date(),
        {_nb_path_sql},
        {_cl_id_sql}
    )
""")

print(f"\U0001f4cb Audit record inserted:")
print(f"   run_id          = {_run_id}")
print(f"   run_type        = {_run_type}")
print(f"   source_rows     = {_source_row_count:,}")
print(f"   eligible_rows   = {_eligible_row_count:,}")
print(f"   changed/predict = {_rows_to_predict:,}")